In [3]:
import os 
from langchain_core.prompts import ChatMessagePromptTemplate,ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import OllamaLLM
from langchain_core.runnables import RunnablePassthrough,RunnableBranch,Runnable,RunnableParallel

In [4]:
import asyncio
from typing import Optional

In [5]:
def get_llm():
    return OllamaLLM(
        model="tinyllama:1.1b",
        temperature=0
    )

llm=get_llm()

#### These three chains represents disticts tasks that can be executed in parallel.

In [6]:
summarize_chain:Runnable=(
    ChatPromptTemplate.from_messages(
        [
            ('system','summarize the following topic conscisely'),
            ('user','{topic}')
        ]
    )
    |llm
    |StrOutputParser()
)

In [9]:
questions_chain:Runnable=(
    ChatPromptTemplate.from_messages([
        
        ('system','Generate three interesting questions about the following topic:'),
        ('user','{topic}')
        
])
    |llm
    |StrOutputParser()
                         
)

In [8]:
terms_chain:Runnable=(
    ChatPromptTemplate.from_messages(
        [
            ('system',"Identify 5-10 keys terms from the follwing topic, separated by commas:"),
            ('user',"{topic}")
        ]
    )
    |llm
    |StrOutputParser()
)

#### Build the parallel + synthesis Chain

##### 1. Define the block of tasks  to run in parallel. the results of these
##### along with the original topic, will be fed into the next step

In [13]:
map_chain=RunnableParallel({
    'summary':summarize_chain,
    'questions':questions_chain,
    'key_terms':terms_chain,
    'topic':RunnablePassthrough(), #Pass the original topic through
})

##### 2. Define the final synthesis prompt which will combine the parallel results.

In [11]:
synthesis_prompt=ChatPromptTemplate.from_messages([
    ('system',""" Based on the following information:
     summary:{summary}
     Related questions:{questions}
     key terms: {key_terms}
     Synthesize a comprehensive answer. """),
    ('user',"Original topic :{topic}")
])

##### 3. Construct the full chain by piping the parallel results directly
##### into the synthesis prompt , followed by the LLM and output parser

In [14]:
full_parallel_chain=map_chain|synthesis_prompt|llm|StrOutputParser()

##### 4. Run the chain

In [15]:
async def run_parallel_example(topic:str)->None:
    """
    Asynchronously invokes the parallel processing chain with a specific topic
    and prints the synthesized results.

    Args:
        topic : The input topic to be processed by the langchain chains.
    """
    
    print(f'\n--- Running Parallel Langchan Example for Topic: "{topic}" --- ')
    
    try:
        #the input to  ainvoke is the single topic stirng
        # then passed to each runnable in the map chain
        response =await full_parallel_chain.ainvoke(topic)
        print(f'======== FINAL RESPONSE ========')
        print(response)
    except Exception as e:
        print(f'\n An error occured during chain execution: {e}')
    
    

In [18]:
if __name__ == '__main__':
    test_topic='the history of space exploration'
    
    await run_parallel_example(test_topic)


--- Running Parallel Langchan Example for Topic: "the history of space exploration" --- 
======== FINAL RESPONSE ========
System:   Based on the following information:

Introduction:
The history of space exploration is an exciting and fascinating one, with many milestones along the way. From the first manned spaceflight to the current ISSES mission, humanity has made significant progress in our understanding of the universe and ourselves. As we continue to explore new frontiers, we can look forward to even more exciting milestones in the future.

Milestone 1: The First Manned Spaceflight (1961)
The first manned spaceflight occurred on July 20th, 1961, when Soviet cosmonaut Yuri Gagarin became the first human to orbit Earth. This was a significant milestone in space exploration as it marked the beginning of human spaceflight and paved the way for future missionss.

Milestone 2: The First Space Shuttle Launch (1983)
In 1983, NASA launched its first space shuttlle, which was named Columb